<a href="https://colab.research.google.com/github/QUANHONGLE/CS421-emotion-prediction/blob/main/notebooks/Q1_ANN_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim
import gensim.downloader as api
from gensim.models import KeyedVectors


import torch
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, classification_report



# Load data directly from GitHub
train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
test_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')
dev_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Dev:", dev_df.shape)
print(train_df.columns.tolist())
print(train_df.head(3))


In [ ]:
# Load pretrained GloVe vectors via gensim
print("Loading GloVe model")
glove_model = api.load("glove-wiki-gigaword-100")  # 100-dimensional GloVe vectors
print("GloVe loaded")

def get_glove_embedding(text, model, dim=100):
    tokens = str(text).lower().split()
    vectors = [model[word] for word in tokens if word in model]
    if len(vectors) == 0:
        return np.zeros(dim)  # if no words found, return zeros
    return np.mean(vectors, axis=0)  # average all word vectors

# Apply to all splits
train_df['glove_emb'] = train_df['text'].apply(lambda x: get_glove_embedding(x, glove_model))
dev_df['glove_emb'] = dev_df['text'].apply(lambda x: get_glove_embedding(x, glove_model))
test_df['glove_emb'] = test_df['text'].apply(lambda x: get_glove_embedding(x, glove_model))

print("GloVe embeddings finished")
print("Sample embedding shape:", train_df['glove_emb'][0].shape)

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Sentence Transformer model")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # small but effective model
print("Sentence Transformer loaded")

# Apply to all splits
train_df['sbert_emb'] = list(sbert_model.encode(train_df['text'].astype(str).tolist(), show_progress_bar=True))
dev_df['sbert_emb'] = list(sbert_model.encode(dev_df['text'].astype(str).tolist(), show_progress_bar=True))
test_df['sbert_emb'] = list(sbert_model.encode(test_df['text'].astype(str).tolist(), show_progress_bar=True))

print("SBERT embeddings finished")
print("Sample embedding shape:", train_df['sbert_emb'][0].shape)

In [ ]:
import torch.nn as nn

class EmotionANN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3, num_polarity_classes=3):
        super(EmotionANN, self).__init__()

        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Task-specific heads
        self.emotion_head = nn.Linear(hidden_dim // 2, 1) # regression
        self.polarity_head = nn.Linear(hidden_dim // 2, num_polarity_classes) # classification
        self.empathy_head = nn.Linear(hidden_dim // 2, 1) # regression

    def forward(self, x):
        shared_out = self.shared(x)
        emotion = self.emotion_head(shared_out).squeeze(1)
        polarity = self.polarity_head(shared_out)
        empathy = self.empathy_head(shared_out).squeeze(1)
        return emotion, polarity, empathy

# Test it works
model_glove = EmotionANN(input_dim=100, num_polarity_classes=4) # GloVe is 100-dim
model_sbert = EmotionANN(input_dim=384, num_polarity_classes=4) # SBERT is 384-dim
print("GloVe ANN:", model_glove)
print("\nSBERT ANN:", model_sbert)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class EmotionDataset(Dataset):
    def __init__(self, df, emb_col, is_test=False):
        self.embeddings = torch.tensor(np.stack(df[emb_col].values), dtype=torch.float32)
        self.is_test = is_test
        if not is_test:
            self.emotion = torch.tensor(df['Emotion'].values, dtype=torch.float32)
            self.polarity = torch.tensor(df['EmotionalPolarity'].values - 1, dtype=torch.long)  # shift to 0-indexed
            self.empathy = torch.tensor(df['Empathy'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.is_test:
            return self.embeddings[idx]
        return self.embeddings[idx], self.emotion[idx], self.polarity[idx], self.empathy[idx]

# Create datasets for GloVe
train_glove_ds = EmotionDataset(train_df, 'glove_emb')
dev_glove_ds   = EmotionDataset(dev_df, 'glove_emb')
test_glove_ds  = EmotionDataset(test_df, 'glove_emb', is_test=True)

# Create datasets for SBERT
train_sbert_ds = EmotionDataset(train_df, 'sbert_emb')
dev_sbert_ds   = EmotionDataset(dev_df, 'sbert_emb')
test_sbert_ds  = EmotionDataset(test_df, 'sbert_emb', is_test=True)

# Dataloaders
train_glove_dl = DataLoader(train_glove_ds, batch_size=32, shuffle=True)
dev_glove_dl   = DataLoader(dev_glove_ds, batch_size=32)
test_glove_dl  = DataLoader(test_glove_ds, batch_size=32)

train_sbert_dl = DataLoader(train_sbert_ds, batch_size=32, shuffle=True)
dev_sbert_dl   = DataLoader(dev_sbert_ds, batch_size=32)
test_sbert_dl  = DataLoader(test_sbert_ds, batch_size=32)

print("Datasets ready!")
print(f"Train size: {len(train_glove_ds)}, Dev size: {len(dev_glove_ds)}, Test size: {len(test_glove_ds)}")

In [ ]:
import torch.optim as optim

def train_model(model, train_dl, dev_dl, epochs=10, lr=1e-3):
    criterion_reg = nn.MSELoss()
    criterion_cls = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        total_loss = 0
        for batch in train_dl:
            embeddings, emotion, polarity, empathy = batch
            optimizer.zero_grad()

            pred_emotion, pred_polarity, pred_empathy = model(embeddings)

            loss_emotion  = criterion_reg(pred_emotion, emotion)
            loss_polarity = criterion_cls(pred_polarity, polarity)
            loss_empathy  = criterion_reg(pred_empathy, empathy)
            loss = loss_emotion + loss_polarity + loss_empathy

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # --- Evaluation on dev set ---
        model.eval()
        all_emotion, all_polarity, all_empathy = [], [], []
        true_emotion, true_polarity, true_empathy = [], [], []

        with torch.no_grad():
            for batch in dev_dl:
                embeddings, emotion, polarity, empathy = batch
                pred_emotion, pred_polarity, pred_empathy = model(embeddings)

                all_emotion.extend(pred_emotion.numpy())
                all_polarity.extend(torch.argmax(pred_polarity, dim=1).numpy())
                all_empathy.extend(pred_empathy.numpy())

                true_emotion.extend(emotion.numpy())
                true_polarity.extend(polarity.numpy())
                true_empathy.extend(empathy.numpy())

        mae_emotion  = mean_absolute_error(true_emotion, all_emotion)
        mae_empathy  = mean_absolute_error(true_empathy, all_empathy)
        report       = classification_report(true_polarity, all_polarity, zero_division=0, output_dict=True)
        f1           = report['macro avg']['f1-score']

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.3f} | "
              f"MAE Emotion: {mae_emotion:.3f} | MAE Empathy: {mae_empathy:.3f} | F1 Polarity: {f1:.3f}")

    return model

print("Training GloVe model...")
model_glove = train_model(model_glove, train_glove_dl, dev_glove_dl)

print("\nTraining SBERT model...")
model_sbert = train_model(model_sbert, train_sbert_dl, dev_sbert_dl)

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, emb_col, is_test=False):
        self.embeddings = torch.tensor(np.stack(df[emb_col].values), dtype=torch.float32)
        self.is_test = is_test
        if not is_test:
            self.emotion = torch.tensor(df['Emotion'].values, dtype=torch.float32)
            self.polarity = torch.tensor(df['EmotionalPolarity'].values, dtype=torch.long)  # no shift needed
            self.empathy = torch.tensor(df['Empathy'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.is_test:
            return self.embeddings[idx]
        return self.embeddings[idx], self.emotion[idx], self.polarity[idx], self.empathy[idx]

# Recreate datasets and dataloaders
train_glove_ds = EmotionDataset(train_df, 'glove_emb')
dev_glove_ds   = EmotionDataset(dev_df, 'glove_emb')
test_glove_ds  = EmotionDataset(test_df, 'glove_emb', is_test=True)

train_sbert_ds = EmotionDataset(train_df, 'sbert_emb')
dev_sbert_ds   = EmotionDataset(dev_df, 'sbert_emb')
test_sbert_ds  = EmotionDataset(test_df, 'sbert_emb', is_test=True)

train_glove_dl = DataLoader(train_glove_ds, batch_size=32, shuffle=True)
dev_glove_dl   = DataLoader(dev_glove_ds, batch_size=32)
test_glove_dl  = DataLoader(test_glove_ds, batch_size=32)

train_sbert_dl = DataLoader(train_sbert_ds, batch_size=32, shuffle=True)
dev_sbert_dl   = DataLoader(dev_sbert_ds, batch_size=32)
test_sbert_dl  = DataLoader(test_sbert_ds, batch_size=32)

print("Datasets ready")

In [ ]:
def predict(model, test_dl):
    model.eval()
    all_emotion, all_polarity, all_empathy = [], [], []
    with torch.no_grad():
        for batch in test_dl:
            embeddings = batch
            pred_emotion, pred_polarity, pred_empathy = model(embeddings)
            all_emotion.extend(pred_emotion.numpy())
            all_polarity.extend(torch.argmax(pred_polarity, dim=1).numpy())
            all_empathy.extend(pred_empathy.numpy())
    return all_emotion, all_polarity, all_empathy

# Use SBERT model for final predictions (it performed better)
emotion_preds, polarity_preds, empathy_preds = predict(model_sbert, test_sbert_dl)

# Save to CSV
predictions_df = pd.DataFrame({
    'id': test_df['id'].values,
    'Emotion': emotion_preds,
    'EmotionalPolarity': polarity_preds,
    'Empathy': empathy_preds
})

predictions_df.to_csv('predictions_ann.csv', index=False)
print("Saved predictions_ann.csv!")
print(predictions_df.head())